In [1]:
%config Completer.use_jedi = True

In [2]:
!pip install -U transformers accelerate bitsandbytes peft

   ━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.8/10.7 MB 23.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━ 4.9/10.7 MB 72.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 86.6 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/60.7 MB ? eta -:--:--

   ━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/60.7 MB 240.5 MB/s eta 0:00:01

...

      Successfully uninstalled accelerate-1.12.0


  Attempting uninstall: transformers


    Found existing installation: transformers 5.0.0


    Uninstalling transformers-5.0.0:


      Successfully uninstalled transformers-5.0.0


In [3]:
import torch
from huggingface_hub import login
#from google.colab import userdata
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, BitsAndBytesConfig, DataCollatorForLanguageModeling
#from peft import LoraConfig, PeftModel, get_peft_model
from accelerate import Accelerator
import pandas as pd
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import DataLoader

In [4]:
#login(userdata.get('HF_TOKEN'))
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)

In [5]:
#medical_corpus = load_dataset("EleutherAI/pile","pubmed_central")
#medical_corpus = load_dataset("FreedomIntelligence/ApolloCorpus", split="train/pretrain/medicalBook_en")
medical_corpus = load_dataset(
    "FreedomIntelligence/ApolloCorpus",
    split="medicalBook_en",
    streaming=True
)

README.md: 0.00B [00:00, ?B/s]

In [6]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

#model = model.to("cuda")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [7]:
def tokenize_fn(corpus):
    tokens = tokenizer(corpus["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


medical_dataset_token = medical_corpus.map(tokenize_fn, batched=True, remove_columns=["text"])
#medical_dataset_token = medical_corpus.map(tokenize_fn)

In [8]:
#LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

#Change QLoRA to use Unsloth
# Training arguments
#training_args = TrainingArguments(
#    output_dir="./llama_med_model",
#    per_device_train_batch_size=4,
#    gradient_accumulation_steps=4,
#    learning_rate=2e-4,
#    num_train_epochs=3,
#    logging_steps=10,
#    max_steps=2000,
#    save_steps=200,
#    fp16=False,
#    bf16=False,
    #packing=True,
#    optim="paged_adamw_8bit"
#)

# Trainer
#trainer = Trainer(
#    model=model,
#    train_dataset=medical_dataset_token,
#    processing_class=tokenizer,
#    args=training_args,
#    data_collator=data_collator
#)

trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


In [9]:
accelerator = Accelerator(mixed_precision="fp16")  # or bf16 if T4 supports
model, optimizer= accelerator.prepare(model, torch.optim.AdamW(model.parameters(), lr=2e-4))

In [10]:
train_loader = DataLoader(
    medical_dataset_token,
    batch_size=4,  # per device
    collate_fn = data_collator
    #collate_fn=lambda batch: {
    #    k: torch.tensor([d[k] for d in batch]) for k in batch[0]
    #}
)

num_epochs = 3
gradient_accumulation_steps = 4
device = accelerator.device
model.train()
for epoch in range(num_epochs):
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        with accelerator.accumulate(model):
            outputs = model(**batch)
            loss = outputs.loss
            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()
            
        if step % 10 == 0:
            print(f"Epoch {epoch} Step {step} Loss {loss.item():.4f}")

Epoch 0 Step 0 Loss 2.7185


Epoch 0 Step 10 Loss 2.2835


Epoch 0 Step 20 Loss 2.4860


Epoch 0 Step 30 Loss 2.1268


Epoch 0 Step 40 Loss 2.3435


...

Epoch 0 Step 45250 Loss 2.0938


Epoch 0 Step 45260 Loss 2.0795


Epoch 0 Step 45270 Loss 2.2259


Epoch 0 Step 45280 Loss 2.1938


Epoch 0 Step 45290 Loss 1.9789


In [ ]:
#trainer.train()

#trainer.save_model("llama_med")
#tokenizer.save_pretrained("llama_med")

In [ ]:
accelerator.wait_for_everyone()
unwrapped_model = accelerator.unwrap_model(model)
unwrapped_model.save_pretrained("llama_med")
tokenizer.save_pretrained("llama_med")